In [1]:
import os, time, random, copy, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import FashionMNIST
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from tqdm import tqdm

# ── GPU / Device ──────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device        : {device}")
if torch.cuda.is_available():
    print(f"GPU Name      : {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Running on CPU — consider enabling GPU in Runtime > Change runtime type.")

Device        : cuda
GPU Name      : Tesla T4
GPU Memory    : 15.64 GB


In [2]:
BASE_OUT   = "/content/outputs"
MODEL_DIR  = f"{BASE_OUT}/models"
PLOT_DIR   = f"{BASE_OUT}/plots"
RESULT_DIR = f"{BASE_OUT}/results"

for d in [BASE_OUT, MODEL_DIR, PLOT_DIR, RESULT_DIR]:
    os.makedirs(d, exist_ok=True)

print("Output directories created:")
for d in [BASE_OUT, MODEL_DIR, PLOT_DIR, RESULT_DIR]:
    print(f"  {d}")


GLOBAL_SEED = 42

BASELINE_CONFIG = {
    "epochs"       : 5,
    "batch_size"   : 32,
    "lr"           : 0.001,
    "optimizer"    : "Adam",
    "activation"   : "ReLU",
    "padding"      : "none",
    "num_classes"  : 10,
    "val_fraction" : 0.1,
    "early_stop"   : False,
    "patience"     : 3,
    "use_gpu"      : True,
}

FMNIST_CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

print("\nBaseline Config:")
for k, v in BASELINE_CONFIG.items():
    print(f"  {k:15s}: {v}")

Output directories created:
  /content/outputs
  /content/outputs/models
  /content/outputs/plots
  /content/outputs/results

Baseline Config:
  epochs         : 5
  batch_size     : 32
  lr             : 0.001
  optimizer      : Adam
  activation     : ReLU
  padding        : none
  num_classes    : 10
  val_fraction   : 0.1
  early_stop     : False
  patience       : 3
  use_gpu        : True


In [3]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(GLOBAL_SEED)

def get_activation(name: str) -> nn.Module:
    """Return an activation module by string name."""
    activations = {
        "ReLU"     : nn.ReLU(),
        "LeakyReLU": nn.LeakyReLU(0.1),
        "GELU"     : nn.GELU(),
        "SiLU"     : nn.SiLU(),
    }
    if name not in activations:
        raise ValueError(f"Unknown activation: {name}. Choose from {list(activations.keys())}")
    return activations[name]

def get_optimizer(name: str, params, lr: float) -> optim.Optimizer:
    """Return an optimizer by string name."""
    opts = {
        "Adam"   : optim.Adam(params, lr=lr),
        "SGD"    : optim.SGD(params, lr=lr, momentum=0.9),
        "RMSprop": optim.RMSprop(params, lr=lr),
    }
    if name not in opts:
        raise ValueError(f"Unknown optimizer: {name}. Choose from {list(opts.keys())}")
    return opts[name]


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def compute_epoch_metrics(outputs, targets):
    all_preds  = torch.cat(outputs).argmax(dim=1).numpy()
    all_labels = torch.cat(targets).numpy()
    acc = (all_preds == all_labels).mean() * 100.0
    return acc


def evaluate_metrics(model, loader):

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            preds = model(imgs).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    acc  = (np.array(all_preds) == np.array(all_labels)).mean() * 100.0
    prec = precision_score(all_labels, all_preds, average="macro", zero_division=0) * 100
    rec  = recall_score   (all_labels, all_preds, average="macro", zero_division=0) * 100
    f1   = f1_score       (all_labels, all_preds, average="macro", zero_division=0) * 100
    return acc, prec, rec, f1, all_preds, all_labels


def plot_curves(history: dict, title: str, save_path: str):

    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(title, fontsize=13)

    axes[0].plot(epochs, history["train_loss"], label="Train Loss", marker="o")
    axes[0].plot(epochs, history["val_loss"],   label="Val Loss",   marker="s")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title("Loss Curve"); axes[0].legend(); axes[0].grid(True)

    axes[1].plot(epochs, history["train_acc"], label="Train Acc", marker="o")
    axes[1].plot(epochs, history["val_acc"],   label="Val Acc",   marker="s")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)")
    axes[1].set_title("Accuracy Curve"); axes[1].legend(); axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.close()


def plot_comparison(df: pd.DataFrame, x_col: str, y_col: str,
                    title: str, save_path: str, palette="viridis"):
    fig, ax = plt.subplots(figsize=(8, 5))
    df_sorted = df.sort_values(y_col, ascending=False).reset_index(drop=True)
    sns.barplot(data=df_sorted, x=x_col, y=y_col, palette=palette, ax=ax)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel(x_col); ax.set_ylabel(y_col)
    for bar in ax.patches:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.2,
            f"{bar.get_height():.2f}",
            ha="center", va="bottom", fontsize=9
        )
    plt.tight_layout()
    plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.close()

def save_results(records: list, csv_path: str) -> pd.DataFrame:
    """Convert list of metric dicts to DataFrame and save CSV."""
    df = pd.DataFrame(records)
    df.to_csv(csv_path, index=False)
    print(f"  Saved results → {csv_path}")
    return df

class EarlyStopping:
    def __init__(self, patience: int = 3, min_delta: float = 1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_loss  = None
        self.stop       = False

    def __call__(self, val_loss: float):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        else:
            self.best_loss = val_loss
            self.counter   = 0

print("Helper functions defined ✓")

Helper functions defined ✓


In [4]:
def build_transform(padding_strategy: str = "none") -> transforms.Compose:
    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )

    to_rgb = transforms.Grayscale(num_output_channels=3)

    if padding_strategy == "none":
        spatial = transforms.Resize((224, 224), antialias=True)
        tf = transforms.Compose([
            to_rgb,
            spatial,
            transforms.ToTensor(),
            normalize,
        ])

    elif padding_strategy == "constant":
        pad_size = 6
        tf = transforms.Compose([
            to_rgb,
            transforms.Pad(pad_size, fill=0, padding_mode="constant"),
            transforms.Resize((224, 224), antialias=True),
            transforms.ToTensor(),
            normalize,
        ])

    elif padding_strategy == "reflect":
        pad_size = 6
        tf = transforms.Compose([
            to_rgb,
            transforms.Pad(pad_size, padding_mode="reflect"),
            transforms.Resize((224, 224), antialias=True),
            transforms.ToTensor(),
            normalize,
        ])
    else:
        raise ValueError(f"Unknown padding strategy: {padding_strategy}")

    return tf


def load_datasets(padding_strategy: str = "none", val_fraction: float = 0.1,
                  seed: int = 42):
    tf = build_transform(padding_strategy)

    full_train = FashionMNIST(root="/content/data", train=True,
                               download=True, transform=tf)
    test_set   = FashionMNIST(root="/content/data", train=False,
                               download=True, transform=tf)


    n_val   = int(len(full_train) * val_fraction)
    n_train = len(full_train) - n_val
    generator = torch.Generator().manual_seed(seed)
    train_set, val_set = random_split(full_train, [n_train, n_val],
                                       generator=generator)

    print(f"  Train samples : {len(train_set):,}")
    print(f"  Val samples   : {len(val_set):,}")
    print(f"  Test samples  : {len(test_set):,}")
    return train_set, val_set, test_set


print("Loading Fashion-MNIST (padding=none)…")
_tr, _va, _te = load_datasets()
print("Dataset loaded ✓")

Loading Fashion-MNIST (padding=none)…


100%|██████████| 26.4M/26.4M [00:02<00:00, 13.1MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 205kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.80MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 14.3MB/s]

  Train samples : 54,000
  Val samples   : 6,000
  Test samples  : 10,000
Dataset loaded ✓


In [5]:
def build_loaders(batch_size: int = 32,
                  padding_strategy: str = "none",
                  val_fraction: float = 0.1,
                  seed: int = 42,
                  num_workers: int = 2):

    train_set, val_set, test_set = load_datasets(
        padding_strategy=padding_strategy,
        val_fraction=val_fraction,
        seed=seed
    )

    train_loader = DataLoader(train_set, batch_size=batch_size,
                               shuffle=True,  num_workers=num_workers,
                               pin_memory=True)
    val_loader   = DataLoader(val_set,   batch_size=batch_size,
                               shuffle=False, num_workers=num_workers,
                               pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=batch_size,
                               shuffle=False, num_workers=num_workers,
                               pin_memory=True)
    return train_loader, val_loader, test_loader


print("DataLoader builder defined ✓")
print("Quick test build:")
_tl, _vl, _tel = build_loaders(batch_size=32)
print(f"  Train batches : {len(_tl)}")
print(f"  Val batches   : {len(_vl)}")
print(f"  Test batches  : {len(_tel)}")

DataLoader builder defined ✓
Quick test build:
  Train samples : 54,000
  Val samples   : 6,000
  Test samples  : 10,000
  Train batches : 1688
  Val batches   : 188
  Test batches  : 313


In [6]:
def build_model(num_classes: int = 10,
                activation: str = "ReLU",
                freeze_backbone: bool = False) -> nn.Module:

    weights = EfficientNet_B0_Weights.IMAGENET1K_V1
    model   = efficientnet_b0(weights=weights)

    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False

    in_features = model.classifier[1].in_features
    act         = get_activation(activation)

    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(in_features, 256),
        act,
        nn.Linear(256, num_classes),
    )

    model = model.to(device)
    return model

_m = build_model(activation="ReLU")
print(f"Trainable parameters : {count_parameters(_m):,}")
print(f"Classifier head      :\n{_m.classifier}")
print("Model builder defined ✓")
del _m

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 107MB/s] 


Trainable parameters : 4,338,054
Classifier head      :
Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=256, bias=True)
  (2): ReLU()
  (3): Linear(in_features=256, out_features=10, bias=True)
)
Model builder defined ✓


In [7]:
def run_experiment(exp_name: str, config: dict) -> tuple:
    set_seed(GLOBAL_SEED)

    # Unpack config
    epochs       = config["epochs"]
    batch_size   = config["batch_size"]
    lr           = config["lr"]
    opt_name     = config["optimizer"]
    act_name     = config["activation"]
    padding      = config["padding"]
    num_classes  = config["num_classes"]
    val_fraction = config["val_fraction"]
    use_early    = config["early_stop"]
    patience     = config["patience"]

    print(f"\n{'='*60}")
    print(f"Experiment : {exp_name}")
    print(f"  lr={lr}, bs={batch_size}, opt={opt_name}, "
          f"act={act_name}, pad={padding}, epochs={epochs}")
    print(f"{'='*60}")

    train_loader, val_loader, test_loader = build_loaders(
        batch_size=batch_size,
        padding_strategy=padding,
        val_fraction=val_fraction,
        seed=GLOBAL_SEED
    )

    model = build_model(num_classes=num_classes, activation=act_name)
    optimizer = get_optimizer(opt_name, model.parameters(), lr)
    criterion = nn.CrossEntropyLoss()

    if use_early:
        stopper = EarlyStopping(patience=patience)

    n_params = count_parameters(model)
    best_val_acc  = 0.0
    best_state    = None

    history = {
        "train_loss": [], "val_loss": [],
        "train_acc" : [], "val_acc" : [],
    }

    start_time = time.time()


    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        epoch_outputs, epoch_targets = [], []

        for imgs, labels in tqdm(train_loader,
                                  desc=f"Epoch {epoch}/{epochs} [Train]",
                                  leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss   = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * imgs.size(0)
            epoch_outputs.append(logits.detach().cpu())
            epoch_targets.append(labels.cpu())

        train_loss = running_loss / len(train_loader.dataset)
        train_acc  = compute_epoch_metrics(epoch_outputs, epoch_targets)

        model.eval()
        val_loss_sum = 0.0
        val_outputs, val_targets = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                logits = model(imgs)
                loss   = criterion(logits, labels)
                val_loss_sum += loss.item() * imgs.size(0)
                val_outputs.append(logits.cpu())
                val_targets.append(labels.cpu())

        val_loss = val_loss_sum / len(val_loader.dataset)
        val_acc  = compute_epoch_metrics(val_outputs, val_targets)

        history["train_loss"].append(round(train_loss, 4))
        history["val_loss"  ].append(round(val_loss,   4))
        history["train_acc" ].append(round(train_acc,  4))
        history["val_acc"   ].append(round(val_acc,    4))

        print(f"  Epoch {epoch:2d} | "
              f"train_loss={train_loss:.4f}  train_acc={train_acc:.2f}%  |  "
              f"val_loss={val_loss:.4f}  val_acc={val_acc:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = copy.deepcopy(model.state_dict())

        if use_early:
            stopper(val_loss)
            if stopper.stop:
                print(f"  Early stopping at epoch {epoch}.")
                break

    total_time = time.time() - start_time


    model_path = f"{MODEL_DIR}/{exp_name}.pth"
    torch.save(best_state, model_path)
    print(f"  Best model saved → {model_path}")


    model.load_state_dict(best_state)
    test_acc, prec, rec, f1, preds, labels_true = evaluate_metrics(model, test_loader)
    print(f"  Test  acc={test_acc:.2f}%  prec={prec:.2f}%  rec={rec:.2f}%  f1={f1:.2f}%")

    curve_path = f"{PLOT_DIR}/{exp_name}_curves.png"
    plot_curves(history,
                title=f"Learning Curves — {exp_name}",
                save_path=curve_path)

    metrics = {
        "experiment"        : exp_name,
        "lr"                : lr,
        "batch_size"        : batch_size,
        "optimizer"         : opt_name,
        "activation"        : act_name,
        "padding"           : padding,
        "epochs_run"        : len(history["train_loss"]),
        "best_val_acc"      : round(best_val_acc,  4),
        "final_train_loss"  : history["train_loss"][-1],
        "final_val_loss"    : history["val_loss"  ][-1],
        "test_accuracy"     : round(test_acc,      4),
        "macro_precision"   : round(prec,          4),
        "macro_recall"      : round(rec,           4),
        "macro_f1"          : round(f1,            4),
        "training_time_s"   : round(total_time,    2),
        "trainable_params"  : n_params,
    }

    return metrics, history


print("Training function defined ✓")

Training function defined ✓


In [10]:
import torch.optim as optim
import torch.nn as nn

def get_optimizer(name: str, params, lr: float) -> optim.Optimizer:
    """Return an optimizer by string name."""
    # Convert params iterator to a list so it can be iterated over multiple times
    params_list = list(params)
    if not params_list:
        raise ValueError("Optimizer received an empty parameter list. This usually means no parameters require gradients.")

    opts = {
        "Adam"   : optim.Adam(params_list, lr=lr),
        "SGD"    : optim.SGD(params_list, lr=lr, momentum=0.9),
        "RMSprop": optim.RMSprop(params_list, lr=lr),
    }
    if name not in opts:
        raise ValueError(f"Unknown optimizer: {name}. Choose from {list(opts.keys())}")
    return opts[name]


LR_VALUES = [0.01, 0.001, 0.0001]

def run_lr_study():
    records = []
    for lr in LR_VALUES:
        cfg = {**BASELINE_CONFIG, "lr": lr}
        exp_name = f"lr_{lr}"
        metrics, history = run_experiment(exp_name, cfg)
        records.append(metrics)

    df = save_results(records, f"{RESULT_DIR}/lr_study.csv")

    df["lr_label"] = df["lr"].astype(str)

    plot_comparison(df, x_col="lr_label", y_col="test_accuracy",
                    title="Learning Rate vs Test Accuracy",
                    save_path=f"{PLOT_DIR}/lr_test_accuracy.png")

    plot_comparison(df, x_col="lr_label", y_col="macro_f1",
                    title="Learning Rate vs Macro F1",
                    save_path=f"{PLOT_DIR}/lr_macro_f1.png")

    print("\nLearning Rate Study Results:")
    print(df[["experiment", "lr", "best_val_acc",
              "test_accuracy", "macro_f1", "training_time_s"]].to_string(index=False))
    return df

lr_df = run_lr_study()


Experiment : lr_0.01
  lr=0.01, bs=32, opt=Adam, act=ReLU, pad=none, epochs=5
  Train samples : 54,000
  Val samples   : 6,000
  Test samples  : 10,000


  Epoch  1 | train_loss=0.5422  train_acc=80.73%  |  val_loss=0.3882  val_acc=86.63%


  Epoch  2 | train_loss=0.3620  train_acc=87.27%  |  val_loss=0.3527  val_acc=88.32%


  Epoch  3 | train_loss=0.3226  train_acc=88.89%  |  val_loss=0.3043  val_acc=89.63%


  Epoch  4 | train_loss=0.2959  train_acc=89.99%  |  val_loss=0.2712  val_acc=90.80%


...

  Epoch  1 | train_loss=0.3467  train_acc=88.42%  |  val_loss=0.1860  val_acc=93.23%


  Epoch  2 | train_loss=0.1800  train_acc=93.65%  |  val_loss=0.1636  val_acc=94.20%


  Epoch  3 | train_loss=0.1387  train_acc=95.03%  |  val_loss=0.1584  val_acc=94.43%


  Epoch  4 | train_loss=0.1121  train_acc=95.94%  |  val_loss=0.1562  val_acc=94.67%


  Epoch  5 | train_loss=0.0887  train_acc=96.84%  |  val_loss=0.1720  val_acc=94.50%
  Best model saved → /content/outputs/models/lr_0.0001.pth
  Test  acc=94.31%  prec=94.31%  rec=94.31%  f1=94.28%
  Saved results → /content/outputs/results/lr_study.csv

Learning Rate Study Results:
experiment     lr  best_val_acc  test_accuracy  macro_f1  training_time_s
   lr_0.01 0.0100       90.8000          89.68   89.6490          1524.55
  lr_0.001 0.0010       94.2833          93.57   93.5594          1525.53
 lr_0.0001 0.0001       94.6667          94.31   94.2785          1515.00


In [11]:
# Cell 9: Batch Size Hyperparameter Study

BS_VALUES = [16, 32, 64]

def run_bs_study():
    records = []
    for bs in BS_VALUES:
        cfg = {**BASELINE_CONFIG, "batch_size": bs}
        exp_name = f"bs_{bs}"
        metrics, history = run_experiment(exp_name, cfg)
        records.append(metrics)

    df = save_results(records, f"{RESULT_DIR}/bs_study.csv")

    df["bs_label"] = df["batch_size"].astype(str)

    plot_comparison(df, x_col="bs_label", y_col="test_accuracy",
                    title="Batch Size vs Test Accuracy",
                    save_path=f"{PLOT_DIR}/bs_test_accuracy.png",
                    palette="magma")

    plot_comparison(df, x_col="bs_label", y_col="macro_f1",
                    title="Batch Size vs Macro F1",
                    save_path=f"{PLOT_DIR}/bs_macro_f1.png",
                    palette="magma")

    print("\nBatch Size Study Results:")
    print(df[["experiment", "batch_size", "best_val_acc",
              "test_accuracy", "macro_f1", "training_time_s"]].to_string(index=False))
    return df

bs_df = run_bs_study()


Experiment : bs_16
  lr=0.001, bs=16, opt=Adam, act=ReLU, pad=none, epochs=5
  Train samples : 54,000
  Val samples   : 6,000
  Test samples  : 10,000


  Epoch  1 | train_loss=0.3559  train_acc=87.55%  |  val_loss=0.2811  val_acc=90.33%


  Epoch  2 | train_loss=0.2462  train_acc=91.25%  |  val_loss=0.2035  val_acc=92.83%


  Epoch  3 | train_loss=0.2065  train_acc=92.68%  |  val_loss=0.2009  val_acc=92.60%


  Epoch  4 | train_loss=0.1861  train_acc=93.36%  |  val_loss=0.1854  val_acc=93.42%


...

  Epoch  1 | train_loss=0.3047  train_acc=89.33%  |  val_loss=0.2271  val_acc=91.52%


  Epoch  2 | train_loss=0.1975  train_acc=93.02%  |  val_loss=0.1677  val_acc=93.88%


  Epoch  3 | train_loss=0.1670  train_acc=94.11%  |  val_loss=0.2389  val_acc=91.87%


  Epoch  4 | train_loss=0.1486  train_acc=94.69%  |  val_loss=0.1757  val_acc=93.67%


  Epoch  5 | train_loss=0.1351  train_acc=95.18%  |  val_loss=0.1765  val_acc=93.85%
  Best model saved → /content/outputs/models/bs_64.pth
  Test  acc=93.74%  prec=93.71%  rec=93.74%  f1=93.71%
  Saved results → /content/outputs/results/bs_study.csv

Batch Size Study Results:
experiment  batch_size  best_val_acc  test_accuracy  macro_f1  training_time_s
     bs_16          16       93.4167          93.22   93.1789          1636.86
     bs_32          32       94.2833          93.57   93.5594          1516.07
     bs_64          64       93.8833          93.74   93.7062          1509.30


In [ ]:
# Cell 10: Optimizer Hyperparameter Study

OPT_VALUES = ["SGD", "Adam", "RMSprop"]

def run_optimizer_study():
    records = []
    for opt_name in OPT_VALUES:
        cfg = {**BASELINE_CONFIG, "optimizer": opt_name}
        exp_name = f"opt_{opt_name}"
        metrics, history = run_experiment(exp_name, cfg)
        records.append(metrics)

    df = save_results(records, f"{RESULT_DIR}/optimizer_study.csv")

    plot_comparison(df, x_col="optimizer", y_col="test_accuracy",
                    title="Optimizer vs Test Accuracy",
                    save_path=f"{PLOT_DIR}/opt_test_accuracy.png",
                    palette="coolwarm")

    plot_comparison(df, x_col="optimizer", y_col="macro_f1",
                    title="Optimizer vs Macro F1",
                    save_path=f"{PLOT_DIR}/opt_macro_f1.png",
                    palette="coolwarm")

    print("\nOptimizer Study Results:")
    print(df[["experiment", "optimizer", "best_val_acc",
              "test_accuracy", "macro_f1", "training_time_s"]].to_string(index=False))
    return df

opt_df = run_optimizer_study()


Experiment : opt_SGD
  lr=0.001, bs=32, opt=SGD, act=ReLU, pad=none, epochs=5
  Train samples : 54,000
  Val samples   : 6,000
  Test samples  : 10,000


  Epoch  1 | train_loss=0.7245  train_acc=77.87%  |  val_loss=0.2862  val_acc=89.52%


  Epoch  2 | train_loss=0.2779  train_acc=90.21%  |  val_loss=0.2219  val_acc=92.18%


  Epoch  3 | train_loss=0.2226  train_acc=92.14%  |  val_loss=0.1922  val_acc=93.25%


  Epoch  4 | train_loss=0.1922  train_acc=93.19%  |  val_loss=0.1833  val_acc=93.47%


  Epoch  5 | train_loss=0.1719  train_acc=93.87%  |  val_loss=0.1744  val_acc=93.98%
  Best model saved → /content/outputs/models/opt_SGD.pth
  Test  acc=93.48%  prec=93.46%  rec=93.48%  f1=93.45%

Experiment : opt_Adam
  lr=0.001, bs=32, opt=Adam, act=ReLU, pad=none, epochs=5
  Train samples : 54,000
  Val samples   : 6,000
  Test samples  : 10,000


Epoch 1/5 [Train]:  12%|█▏        | 208/1688 [00:36<04:14,  5.82it/s]

In [ ]:
# Cell 11: Epoch Count Hyperparameter Study

EPOCH_VALUES = [3, 5, 8]

def run_epoch_study():
    records = []
    for ep in EPOCH_VALUES:
        cfg = {**BASELINE_CONFIG, "epochs": ep}
        exp_name = f"epochs_{ep}"
        metrics, history = run_experiment(exp_name, cfg)
        records.append(metrics)

    df = save_results(records, f"{RESULT_DIR}/epoch_study.csv")

    df["epoch_label"] = df["epochs_run"].astype(str)

    plot_comparison(df, x_col="epoch_label", y_col="test_accuracy",
                    title="Epoch Count vs Test Accuracy",
                    save_path=f"{PLOT_DIR}/epoch_test_accuracy.png",
                    palette="cividis")

    plot_comparison(df, x_col="epoch_label", y_col="macro_f1",
                    title="Epoch Count vs Macro F1",
                    save_path=f"{PLOT_DIR}/epoch_macro_f1.png",
                    palette="cividis")

    print("\nEpoch Count Study Results:")
    print(df[["experiment", "epochs_run", "best_val_acc",
              "test_accuracy", "macro_f1", "training_time_s"]].to_string(index=False))
    return df

epoch_df = run_epoch_study()

In [ ]:
# Cell 12: Activation Function Hyperparameter Study

ACT_VALUES = ["ReLU", "LeakyReLU", "GELU", "SiLU"]

def run_activation_study():
    records = []
    for act in ACT_VALUES:
        cfg = {**BASELINE_CONFIG, "activation": act}
        exp_name = f"act_{act}"
        metrics, history = run_experiment(exp_name, cfg)
        records.append(metrics)

    df = save_results(records, f"{RESULT_DIR}/activation_study.csv")

    plot_comparison(df, x_col="activation", y_col="test_accuracy",
                    title="Activation Function vs Test Accuracy",
                    save_path=f"{PLOT_DIR}/act_test_accuracy.png",
                    palette="plasma")

    plot_comparison(df, x_col="activation", y_col="macro_f1",
                    title="Activation Function vs Macro F1",
                    save_path=f"{PLOT_DIR}/act_macro_f1.png",
                    palette="plasma")

    print("\nActivation Study Results:")
    print(df[["experiment", "activation", "best_val_acc",
              "test_accuracy", "macro_f1", "training_time_s"]].to_string(index=False))
    return df

act_df = run_activation_study()

In [ ]:
# Cell 13: Preprocessing Padding Strategy Study

PAD_VALUES = ["none", "constant", "reflect"]

def run_padding_study():
    records = []
    for pad in PAD_VALUES:
        cfg = {**BASELINE_CONFIG, "padding": pad}
        exp_name = f"pad_{pad}"
        metrics, history = run_experiment(exp_name, cfg)
        records.append(metrics)

    df = save_results(records, f"{RESULT_DIR}/padding_study.csv")

    plot_comparison(df, x_col="padding", y_col="test_accuracy",
                    title="Padding Strategy vs Test Accuracy",
                    save_path=f"{PLOT_DIR}/pad_test_accuracy.png",
                    palette="mako")

    plot_comparison(df, x_col="padding", y_col="macro_f1",
                    title="Padding Strategy vs Macro F1",
                    save_path=f"{PLOT_DIR}/pad_macro_f1.png",
                    palette="mako")

    print("\nPadding Strategy Study Results:")
    print(df[["experiment", "padding", "best_val_acc",
              "test_accuracy", "macro_f1", "training_time_s"]].to_string(index=False))
    return df

pad_df = run_padding_study()

In [ ]:
# Cell 14: Combine All Results → Master CSV

all_dfs = [lr_df, bs_df, opt_df, epoch_df, act_df, pad_df]
master_df = pd.concat(all_dfs, ignore_index=True)

# Drop duplicate baseline experiments if any (keep first occurrence)
master_df = master_df.drop_duplicates(subset="experiment", keep="first")

master_csv = f"{RESULT_DIR}/master_results.csv"
master_df.to_csv(master_csv, index=False)
print(f"Master CSV saved → {master_csv}")

# Sort by best_val_acc descending
master_sorted = master_df.sort_values(
    ["best_val_acc", "macro_f1"], ascending=[False, False]
).reset_index(drop=True)

print("\n── All Experiments Sorted by Validation Accuracy ──")
display_cols = [
    "experiment", "lr", "batch_size", "optimizer",
    "activation", "padding", "epochs_run",
    "best_val_acc", "test_accuracy", "macro_f1", "training_time_s"
]
print(master_sorted[display_cols].to_string(index=False))

# Identify best overall
best_row = master_sorted.iloc[0]
print(f"\n🏆 Best Experiment  : {best_row['experiment']}")
print(f"   Val Accuracy     : {best_row['best_val_acc']:.2f}%")
print(f"   Test Accuracy    : {best_row['test_accuracy']:.2f}%")
print(f"   Macro F1         : {best_row['macro_f1']:.2f}%")

In [ ]:
# Cell 15: Reload Best Model → Confusion Matrix + Classification Report

best_exp_name = master_sorted.iloc[0]["experiment"]
best_cfg_row  = master_sorted.iloc[0]

# Reconstruct config for the best experiment
best_cfg = {
    "epochs"       : int(best_cfg_row["epochs_run"]),
    "batch_size"   : int(best_cfg_row["batch_size"]),
    "lr"           : float(best_cfg_row["lr"]),
    "optimizer"    : best_cfg_row["optimizer"],
    "activation"   : best_cfg_row["activation"],
    "padding"      : best_cfg_row["padding"],
    "num_classes"  : BASELINE_CONFIG["num_classes"],
    "val_fraction" : BASELINE_CONFIG["val_fraction"],
    "early_stop"   : BASELINE_CONFIG["early_stop"],
    "patience"     : BASELINE_CONFIG["patience"],
    "use_gpu"      : BASELINE_CONFIG["use_gpu"],
}

print(f"Evaluating best experiment: {best_exp_name}")
print(f"  activation={best_cfg['activation']}, padding={best_cfg['padding']}")

# ── Reload model ──────────────────────────────────────────────────────────────
best_model_path = f"{MODEL_DIR}/{best_exp_name}.pth"
best_model = build_model(num_classes=best_cfg["num_classes"],
                          activation=best_cfg["activation"])
best_model.load_state_dict(torch.load(best_model_path, map_location=device))
best_model.eval()
print(f"Model loaded from {best_model_path}")

# ── Test loader for best config ───────────────────────────────────────────────
_, _, test_loader_best = build_loaders(
    batch_size=best_cfg["batch_size"],
    padding_strategy=best_cfg["padding"],
    val_fraction=best_cfg["val_fraction"],
    seed=GLOBAL_SEED
)

# ── Full evaluation ───────────────────────────────────────────────────────────
test_acc, prec, rec, f1, all_preds, all_labels = evaluate_metrics(
    best_model, test_loader_best
)
print(f"\nFinal Test Metrics:")
print(f"  Accuracy  : {test_acc:.2f}%")
print(f"  Precision : {prec:.2f}%")
print(f"  Recall    : {rec:.2f}%")
print(f"  F1 Score  : {f1:.2f}%")

# ── Classification report ─────────────────────────────────────────────────────
report = classification_report(all_labels, all_preds,
                                 target_names=FMNIST_CLASSES,
                                 digits=4)
print(f"\nClassification Report:\n{report}")

report_path = f"{RESULT_DIR}/best_model_classification_report.txt"
with open(report_path, "w") as f:
    f.write(f"Best Experiment: {best_exp_name}\n\n")
    f.write(report)
print(f"Report saved → {report_path}")

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=FMNIST_CLASSES,
            yticklabels=FMNIST_CLASSES,
            linewidths=0.5, ax=ax)
ax.set_title(f"Confusion Matrix — {best_exp_name}", fontsize=14, pad=15)
ax.set_xlabel("Predicted Label", fontsize=12)
ax.set_ylabel("True Label",      fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()

cm_path = f"{PLOT_DIR}/best_model_confusion_matrix.png"
plt.savefig(cm_path, dpi=120, bbox_inches="tight")
plt.close()
print(f"Confusion matrix saved → {cm_path}")

In [ ]:
# Cell 16: Final Summary — Best Hyperparameter per Group

print("=" * 65)
print("              FINAL HYPERPARAMETER STUDY SUMMARY")
print("=" * 65)

def best_in_group(df, group_col, rank_col="test_accuracy"):
    """Return the row with the highest rank_col for a given study df."""
    return df.sort_values(rank_col, ascending=False).iloc[0]

# Best per group
b_lr   = best_in_group(lr_df,    "lr")
b_bs   = best_in_group(bs_df,    "batch_size")
b_opt  = best_in_group(opt_df,   "optimizer")
b_ep   = best_in_group(epoch_df, "epochs_run")
b_act  = best_in_group(act_df,   "activation")
b_pad  = best_in_group(pad_df,   "padding")

print(f"\n{'Group':<28} {'Best Value':<14} {'Test Acc':>9}  {'Macro F1':>9}")
print("-" * 65)
print(f"{'Learning Rate':<28} {str(b_lr['lr']):<14} {b_lr['test_accuracy']:>8.2f}%  {b_lr['macro_f1']:>8.2f}%")
print(f"{'Batch Size':<28} {str(int(b_bs['batch_size'])):<14} {b_bs['test_accuracy']:>8.2f}%  {b_bs['macro_f1']:>8.2f}%")
print(f"{'Optimizer':<28} {str(b_opt['optimizer']):<14} {b_opt['test_accuracy']:>8.2f}%  {b_opt['macro_f1']:>8.2f}%")
print(f"{'Epoch Count':<28} {str(int(b_ep['epochs_run'])):<14} {b_ep['test_accuracy']:>8.2f}%  {b_ep['macro_f1']:>8.2f}%")
print(f"{'Activation Function':<28} {str(b_act['activation']):<14} {b_act['test_accuracy']:>8.2f}%  {b_act['macro_f1']:>8.2f}%")
print(f"{'Padding Strategy':<28} {str(b_pad['padding']):<14} {b_pad['test_accuracy']:>8.2f}%  {b_pad['macro_f1']:>8.2f}%")
print("-" * 65)

# Overall best
overall_best = master_sorted.iloc[0]
print(f"\n{'OVERALL BEST EXPERIMENT':<28} {overall_best['experiment']}")
print(f"  lr={overall_best['lr']}, bs={int(overall_best['batch_size'])}, "
      f"opt={overall_best['optimizer']}, act={overall_best['activation']}, "
      f"pad={overall_best['padding']}, epochs={int(overall_best['epochs_run'])}")
print(f"  Test Accuracy : {overall_best['test_accuracy']:.2f}%")
print(f"  Macro F1      : {overall_best['macro_f1']:.2f}%")
print(f"  Val Accuracy  : {overall_best['best_val_acc']:.2f}%")
print("=" * 65)

# ── Code-generated observations ───────────────────────────────────────────────
print("\nAuto-generated Observations:")

# LR observation
lr_vals  = lr_df.sort_values("test_accuracy", ascending=False)["lr"].tolist()
print(f"  • Best LR={lr_vals[0]} outperformed LR={lr_vals[-1]} — "
      f"too-high LR may cause instability, too-low LR underfits in few epochs.")

# Batch size
bs_vals  = bs_df.sort_values("test_accuracy", ascending=False)["batch_size"].tolist()
print(f"  • Batch size {int(bs_vals[0])} showed best accuracy. "
      f"Smaller batches provide noisier but often better-generalizing gradients.")

# Optimizer
opt_vals = opt_df.sort_values("test_accuracy", ascending=False)["optimizer"].tolist()
print(f"  • {opt_vals[0]} optimizer ranked first. "
      f"Adaptive optimizers typically converge faster on transfer learning tasks.")

# Epochs
ep_vals  = epoch_df.sort_values("test_accuracy", ascending=False)["epochs_run"].tolist()
print(f"  • {int(ep_vals[0])} epochs gave best results. "
      f"More epochs generally help unless overfitting occurs.")

# Activation
act_vals = act_df.sort_values("test_accuracy", ascending=False)["activation"].tolist()
print(f"  • {act_vals[0]} activation in the classifier head yielded highest accuracy.")

# Padding
pad_vals = pad_df.sort_values("test_accuracy", ascending=False)["padding"].tolist()
print(f"  • Padding strategy '{pad_vals[0]}' worked best. "
      f"Padding before resize can preserve edge features for small images like MNIST.")

# Outputs summary
print("\nAll outputs saved to:")
print(f"  Models   → {MODEL_DIR}/")
print(f"  Plots    → {PLOT_DIR}/")
print(f"  Results  → {RESULT_DIR}/")
print(f"  Master   → {RESULT_DIR}/master_results.csv")
print("\nStudy complete ✓")